# Meeting-room Gaussian Splat pilot
Reads the successful namas reconstruction. Standard gsplat 1.5.3 rasterization and DefaultStrategy; custom minimal training loop with L1 + SSIM loss. 4,000 iterations at 640px; every eighth view held out of photometric training. Exports PLY, checkpoints, comparisons, and metrics. One T4 is used. Source: https://github.com/nerfstudio-project/gsplat/tree/v1.5.3

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['TORCH_CUDA_ARCH_LIST']='7.5'
os.environ['MAX_JOBS']='2'
os.environ['TORCH_EXTENSIONS_DIR']='/kaggle/working/torch_extensions'
subprocess.run(['nvidia-smi'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','gsplat==1.5.3','pycolmap==4.2.1','pytorch-msssim==1.0.0','ninja'],check=True)


In [ ]:
WORKER = '"""Small reproducible gsplat baseline. Uses gsplat\'s standard densification strategy.\nReference: https://github.com/nerfstudio-project/gsplat/tree/v1.5.3\nInput: saved COLMAP pilot, not a new capture. Coordinates remain in COLMAP units.\n"""\nimport os, json, time, random\nfrom pathlib import Path\nimport cv2, numpy as np, torch, pycolmap\nfrom PIL import Image\nfrom scipy.spatial import cKDTree\nfrom pytorch_msssim import ssim\nfrom gsplat import rasterization, export_splats\nfrom gsplat.strategy import DefaultStrategy\n\ntorch.manual_seed(42); np.random.seed(42); random.seed(42)\ntorch.set_num_threads(4)\nout=Path(\'/kaggle/working/splat_output\'); out.mkdir(exist_ok=True)\nassert torch.cuda.is_available(), \'CUDA unavailable\'\nassert \'T4\' in torch.cuda.get_device_name(0), \'Expected T4\'\nsources=list(Path(\'/kaggle/input\').rglob(\'sfm_report.json\'))\nassert len(sources)==1, f\'Expected one reconstruction, got {sources}\'\nsource=sources[0].parent\nrec=pycolmap.Reconstruction(source/\'sparse/0\')\nassert rec.num_reg_images()==180\nimages=sorted(rec.images.values(),key=lambda i:i.name)\nframes=[]; views=[]; intrinsics=[]; centers=[]; names=[]\nfor im in images:\n    cam=rec.cameras[im.camera_id]\n    assert cam.model.name==\'SIMPLE_RADIAL\', cam.model.name\n    rgb=cv2.imread(str(source/\'images\'/im.name))\n    assert rgb is not None\n    rgb=cv2.cvtColor(rgb,cv2.COLOR_BGR2RGB)\n    h,w=rgb.shape[:2]\n    rgb=cv2.resize(rgb,(640,round(h*640/w)),interpolation=cv2.INTER_AREA)\n    h,w=rgb.shape[:2]\n    K=cam.calibration_matrix().copy(); K[0]*=w/cam.width; K[1]*=h/cam.height\n    dist=np.array([cam.params[3],0.,0.,0.])\n    newK,_=cv2.getOptimalNewCameraMatrix(K,dist,(w,h),0,(w,h))\n    rgb=cv2.undistort(rgb,K,dist,None,newK)\n    view=np.eye(4,dtype=np.float32); view[:3]=im.cam_from_world().matrix()\n    centers.append(np.linalg.inv(view)[:3,3])\n    frames.append(torch.from_numpy(rgb.copy()))\n    views.append(view); intrinsics.append(newK); names.append(im.name)\nviews=torch.tensor(np.stack(views),device=\'cuda\',dtype=torch.float32)\nintrinsics=torch.tensor(np.stack(intrinsics),device=\'cuda\',dtype=torch.float32)\ncenters=np.stack(centers)\nscene_scale=float(np.max(np.linalg.norm(centers-centers.mean(0),axis=1)))\npoints=list(rec.points3D.values())\nxyz=np.array([p.xyz for p in points],dtype=np.float32)\ncolors=np.array([p.color for p in points],dtype=np.float32)/255\ndistances=cKDTree(xyz).query(xyz,k=4)[0][:,1:]\nscales=np.log(np.maximum(np.sqrt((distances**2).mean(1)),1e-6))\ndef param(x): return torch.nn.Parameter(torch.as_tensor(x,dtype=torch.float32,device=\'cuda\'))\nn=len(points)\nparams=torch.nn.ParameterDict({\n    \'means\':param(xyz), \'scales\':param(np.repeat(scales[:,None],3,axis=1)),\n    \'quats\':param(np.random.rand(n,4)), \'opacities\':param(np.full(n,np.log(.1/.9))),\n    \'sh0\':param(((colors-.5)/.28209479177387814)[:,None]), \'shN\':param(np.zeros((n,15,3)))})\nlrs={\'means\':1.6e-4*scene_scale,\'scales\':.005,\'quats\':.001,\'opacities\':.05,\'sh0\':.0025,\'shN\':.0025/20}\noptim={k:torch.optim.Adam([v],lr=lrs[k],eps=1e-15) for k,v in params.items()}\nstrategy=DefaultStrategy(refine_start_iter=300,refine_stop_iter=2500,reset_every=1500,refine_every=100,verbose=False)\nstrategy.check_sanity(params,optim)\nstate=strategy.initialize_state(scene_scale=scene_scale)\ntrain=[i for i in range(180) if i%8!=0]; test=[i for i in range(180) if i%8==0]\nsteps=4000\nconfig={\'steps\':steps,\'width\':w,\'height\':h,\'train_images\':len(train),\'held_out_images\':len(test),\n        \'seed\':42,\'torch\':torch.__version__,\'cuda\':torch.version.cuda,\'gpu\':torch.cuda.get_device_name(0),\n        \'source\':str(source),\'scene_scale\':scene_scale,\'gsplat\':\'1.5.3\',\n        \'evaluation_note\':\'Views held out of photometric training; COLMAP poses and seed geometry used all 180 images.\'}\n(out/\'config.json\').write_text(json.dumps(config,indent=2))\n(out/\'cameras.json\').write_text(json.dumps({\'names\':names,\'world_to_camera\':views.cpu().tolist(),\'intrinsics\':intrinsics.cpu().tolist(),\'width\':w,\'height\':h}))\nprint(\'DATA_READY\',json.dumps(config),flush=True)\ndef render(i,degree):\n    return rasterization(means=params[\'means\'],quats=params[\'quats\'],scales=params[\'scales\'].exp(),\n        opacities=params[\'opacities\'].sigmoid(),colors=torch.cat([params[\'sh0\'],params[\'shN\']],1),\n        viewmats=views[i:i+1],Ks=intrinsics[i:i+1],width=w,height=h,sh_degree=degree,\n        near_plane=.01,far_plane=1e6,packed=False)\ndef export(step):\n    data={k:v.detach().cpu() for k,v in params.items()}\n    assert all(torch.isfinite(v).all() for v in data.values())\n    torch.save({\'step\':step,\'splats\':data,\'config\':config,\'optimizers\':{k:v.state_dict() for k,v in optim.items()},\'strategy_state\':state},out/\'checkpoint.pt\')\n    export_splats(**data,format=\'ply\',save_to=str(out/\'meetingroom.ply\'))\n    print(\'EXPORTED\',step,len(data[\'means\']),flush=True)\nstarted=time.time()\nfor step in range(steps):\n    if step%len(train)==0: random.shuffle(train)\n    i=train[step%len(train)]\n    gt=frames[i].to(\'cuda\',dtype=torch.float32)[None]/255\n    degree=min(step//1000,3)\n    rendered,alpha,info=render(i,degree)\n    strategy.step_pre_backward(params,optim,state,step,info)\n    loss=.8*(rendered-gt).abs().mean()+.2*(1-ssim(rendered.permute(0,3,1,2),gt.permute(0,3,1,2),data_range=1,size_average=True))\n    assert torch.isfinite(loss), \'Non-finite training loss\'\n    loss.backward()\n    for opt in optim.values(): opt.step(); opt.zero_grad(set_to_none=True)\n    optim[\'means\'].param_groups[0][\'lr\']=lrs[\'means\']*(.01**((step+1)/steps))\n    if len(params[\'means\'])>400000: strategy.refine_stop_iter=step\n    strategy.step_post_backward(params,optim,state,step,info,packed=False)\n    if step%100==0:\n        status={\'step\':step,\'loss\':float(loss.detach()),\'splats\':len(params[\'means\']),\'elapsed_seconds\':time.time()-started}\n        (out/\'progress.json\').write_text(json.dumps(status,indent=2)); print(status,flush=True)\n    if step+1 in [1000,4000]: export(step+1)\nmetrics=[]\nwith torch.no_grad():\n    for j,i in enumerate(test):\n        pred=render(i,3)[0].clamp(0,1)\n        gt=frames[i].to(\'cuda\',dtype=torch.float32)[None]/255\n        mse=(pred-gt).square().mean()\n        score={\'image\':names[i],\'psnr\':float(-10*torch.log10(mse)),\n               \'ssim\':float(ssim(pred.permute(0,3,1,2),gt.permute(0,3,1,2),data_range=1,size_average=True))}\n        metrics.append(score)\n        if j%4==0:\n            pair=torch.cat([gt[0],pred[0]],dim=1).cpu().numpy()\n            Image.fromarray((pair*255).astype(np.uint8)).save(out/f\'comparison_{j:02d}.jpg\')\nsummary={\'result\':\'completed\',\'steps\':steps,\'gaussians\':len(params[\'means\']),\n         \'training_and_evaluation_seconds\':time.time()-started,\'psnr_mean\':float(np.mean([m[\'psnr\'] for m in metrics])),\n         \'ssim_mean\':float(np.mean([m[\'ssim\'] for m in metrics])),\'views\':metrics,\n         \'ply_bytes\':(out/\'meetingroom.ply\').stat().st_size,\'max_gpu_allocated_gb\':torch.cuda.max_memory_allocated()/1024**3}\n(out/\'metrics.json\').write_text(json.dumps(summary,indent=2))\nprint(\'TRAINING_COMPLETE\',json.dumps(summary),flush=True)\n'
p=Path('/kaggle/working/train_splat.py')
p.write_text(WORKER)
with open('/kaggle/working/training.log','w') as log:
    result=subprocess.run([sys.executable,'-u',str(p)],stdout=log,stderr=subprocess.STDOUT,timeout=5400)
print(Path('/kaggle/working/training.log').read_text()[-12000:])
assert result.returncode==0, 'Training failed; inspect training.log'
